# Chapter 4: Building Models with Distance Metrics and Nearest Neighbors

## Ringkasan Bab & Penjelasan Teoretis
Bab ini membahas pemodelan berbasis metrik jarak spasial dengan fokus utama pada algoritma **k-Nearest Neighbors (KNN)**.

### 1. Metrik Jarak (*Distance Metrics*) & Algoritma KNN
* **Euclidean Distance**: Jarak garis lurus standar ($L_2$ norm) di ruang berdimensi-$n$: $d(p, q) = \sqrt{\sum (p_i - q_i)^2}$.
* **Manhattan Distance**: Jarak blok kota ($L_1$ norm) yang menghitung jumlah selisih absolut di sepanjang sumbu ortogonal: $d(p, q) = \sum |p_i - q_i|$. Lebih tahan terhadap *outlier*.
* **Minkowski Distance**: Bentuk generalisasi jarak Euclidean ($p=2$) dan Manhattan ($p=1$): $d(p, q) = (\sum |p_i - q_i|^p)^{1/p}$.
* **Mekanisme KNN**: Mengklasifikasikan titik data baru berdasarkan pemungutan suara mayoritas (*majority voting*) dari $k$ tetangga terdekatnya. Nilai $k$ yang kecil sensitif terhadap derau (*overfitting*), sedangkan $k$ yang besar memperhalus batas keputusan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_circles
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, learning_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2024)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print("Accuracy (Basic KNN, k=3):", accuracy_score(y_test, y_pred))

# Perbandingan Metrik Jarak pada Noisy Circles vs Checkerboard Dataset
n_samples = 300
X_circles, y_circles = make_circles(n_samples=n_samples, noise=0.3, factor=0.3, random_state=2024)
X_circles_train, X_circles_test, y_circles_train, y_circles_test = train_test_split(
    X_circles, y_circles, test_size=0.2, random_state=2024
)

x_g = np.linspace(0, 4, int(np.sqrt(n_samples)))
y_g = np.linspace(0, 4, int(np.sqrt(n_samples)))
xx, yy = np.meshgrid(x_g, y_g)
X_moons = np.column_stack((xx.ravel(), yy.ravel()))
y_moons = np.mod(np.floor(xx.ravel()) + np.floor(yy.ravel()), 2)
X_moons_train, X_moons_test, y_moons_train, y_moons_test = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=2024
)

metrics = ['euclidean', 'manhattan', 'minkowski']
results = {'circles': {}, 'moons': {}}

for metric in metrics:
    knn_circles = KNeighborsClassifier(n_neighbors=3, metric=metric).fit(X_circles_train, y_circles_train)
    results['circles'][metric] = accuracy_score(y_circles_test, knn_circles.predict(X_circles_test))

    knn_moons = KNeighborsClassifier(n_neighbors=3, metric=metric).fit(X_moons_train, y_moons_train)
    results['moons'][metric] = accuracy_score(y_moons_test, knn_moons.predict(X_moons_test))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
s1_tr = ax1.scatter(X_circles_train[:, 0], X_circles_train[:, 1], c=y_circles_train, cmap='viridis', alpha=0.6)
s1_te = ax1.scatter(X_circles_test[:, 0], X_circles_test[:, 1], color='red', marker='x', s=50)
ax1.set_title('Noisy Circles Dataset')
lines1 = [plt.Line2D([0], [0], color='white') for _ in metrics]
legend_text1 = [f"{m.capitalize()}: {results['circles'][m]:.3f}" for m in metrics]
ax1.legend(lines1 + [s1_tr, s1_te], legend_text1 + ['Training Points', 'Test Points'],
           title='Accuracy by Metric', bbox_to_anchor=(1.05, 1), loc='upper left')

s2_tr = ax2.scatter(X_moons_train[:, 0], X_moons_train[:, 1], c=y_moons_train, cmap='viridis', alpha=0.6)
s2_te = ax2.scatter(X_moons_test[:, 0], X_moons_test[:, 1], color='red', marker='x', s=50)
ax2.set_title('Checkerboard Dataset')
lines2 = [plt.Line2D([0], [0], color='white') for _ in metrics]
legend_text2 = [f"{m.capitalize()}: {results['moons'][m]:.3f}" for m in metrics]
ax2.legend(lines2 + [s2_tr, s2_te], legend_text2 + ['Training Points', 'Test Points'],
           title='Accuracy by Metric', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 2. Hyperparameter Tuning & Evaluasi Kinerja KNN
* **Grid Search (`GridSearchCV`)**: Mencari kombinasi optimal `n_neighbors`, `weights` (`uniform` vs `distance`), dan `metric` menggunakan *5-fold cross-validation*.
* **Evaluasi Kinerja**: Menggunakan *learning curves* untuk memantau konvergensi pelatihan, *confusion matrix* untuk memetakan prediksi benar/salah per kelas, serta laporan klasifikasi (*precision*, *recall*, *F1-score*).

In [ ]:
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

cv_scores = cross_val_score(grid_search.best_estimator_, X_train, y_train, cv=5)
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean(), "| Std:", cv_scores.std())

train_sizes, train_scores, test_scores = learning_curve(
    grid_search.best_estimator_, X_train, y_train, train_sizes=np.linspace(0.1, 1.0, 10), cv=5
)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, np.mean(train_scores, axis=1), label='Training score')
plt.plot(train_sizes, np.mean(test_scores, axis=1), label='Cross-validation score')
plt.xlabel('Training examples')
plt.ylabel('Score')
plt.legend(loc='best')
plt.title('Learning Curves')
plt.show()

y_pred = grid_search.best_estimator_.predict(X_test)
class_names = iris.target_names

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Number of Samples'})
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

report_df = pd.DataFrame(classification_report(y_test, y_pred, target_names=class_names, output_dict=True)).transpose()
display(report_df.style.background_gradient(cmap='Blues', subset=['precision', 'recall', 'f1-score'])
        .format({'precision': '{:.3f}', 'recall': '{:.3f}', 'f1-score': '{:.3f}', 'support': '{:.0f}'}))